[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/22_robotics_action_policies.ipynb)

# 22. Robotics action policies — paper-faithful tiny structures

목표는 **모델 크기만 줄이고 action-generation 계산 구조는 유지하는 것**이다.

- ACT: action chunk + CVAE latent + Transformer query decoder
- Diffusion Policy: observation-conditioned temporal **1D U-Net**
- π0-style: separate prefix/action-expert parameters + asymmetric joint attention + flow matching
- FAST: quantile normalization → DCT → quantization → BPE

대규모 vision backbone이나 pretrained VLM은 작은 token projection으로 바꾸지만, 핵심 action policy를 단순 MLP나 oracle velocity로 대체하지 않는다.


In [ ]:
import math
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 1. ACT — chunk + CVAE latent + learned action queries

ACT는 demonstration action chunk를 encoder가 읽어 latent distribution을 만들고,
decoder의 여러 learned query가 observation과 latent를 condition으로 받아 미래 action chunk를 동시에 생성한다.


In [ ]:
class TinyACT(nn.Module):
    def __init__(self, obs_dim=8, action_dim=3, chunk=4, hidden=24, latent=8):
        super().__init__()
        self.latent = latent

        self.obs_proj = nn.Linear(obs_dim, hidden)
        self.action_proj = nn.Linear(action_dim, hidden)

        enc = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=3, dim_feedforward=4 * hidden, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc, num_layers=1)
        self.mu = nn.Linear(hidden, latent)
        self.logvar = nn.Linear(hidden, latent)

        self.latent_proj = nn.Linear(latent, hidden)
        self.queries = nn.Parameter(torch.randn(1, chunk, hidden) * 0.02)

        dec = nn.TransformerDecoderLayer(
            d_model=hidden, nhead=3, dim_feedforward=4 * hidden, batch_first=True
        )
        self.decoder = nn.TransformerDecoder(dec, num_layers=1)
        self.action_head = nn.Linear(hidden, action_dim)

    def forward(self, obs, target_chunk=None):
        batch = obs.size(0)
        obs_token = self.obs_proj(obs).unsqueeze(1)

        if target_chunk is not None:
            action_tokens = self.action_proj(target_chunk)
            encoded = self.encoder(torch.cat([obs_token, action_tokens], dim=1))[:, 0]
            mu = self.mu(encoded)
            logvar = self.logvar(encoded)
            z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        else:
            mu = torch.zeros(batch, self.latent, device=obs.device)
            logvar = torch.zeros_like(mu)
            z = torch.zeros_like(mu)

        memory = torch.cat([obs_token, self.latent_proj(z).unsqueeze(1)], dim=1)
        queries = self.queries.expand(batch, -1, -1)
        return self.action_head(self.decoder(queries, memory)), mu, logvar


obs = torch.randn(4, 8, device=device)
target_chunk = torch.randn(4, 4, 3, device=device)

act = TinyACT().to(device)
pred_chunk, mu, logvar = act(obs, target_chunk)

act_loss = F.l1_loss(pred_chunk, target_chunk)
act_loss = act_loss - 0.005 * torch.mean(1 + logvar - mu.square() - logvar.exp())
act_loss.backward()

print("ACT chunk:", pred_chunk.shape)
print("ACT decoder grad:", act.action_head.weight.grad.norm().item())


## 2. Diffusion Policy — conditional temporal U-Net

원 논문 계열의 핵심은 action horizon 전체를 denoise하는 **conditional 1D U-Net**이다.
아래에서는 channel 수와 stage 수만 줄이고 down / bottleneck / up / skip / FiLM-style conditioning을 유지한다.


In [ ]:
def time_embedding(t, dim):
    half = dim // 2
    freq = torch.exp(
        -math.log(10000.0)
        * torch.arange(half, device=t.device, dtype=t.dtype)
        / max(half - 1, 1)
    )
    angle = t[:, None] * freq[None]
    return torch.cat([angle.sin(), angle.cos()], dim=-1)


class CondRes1D(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv1d(out_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(4, out_ch)
        self.norm2 = nn.GroupNorm(4, out_ch)
        self.cond = nn.Linear(cond_dim, 2 * out_ch)
        self.skip = nn.Identity() if in_ch == out_ch else nn.Conv1d(in_ch, out_ch, 1)

    def forward(self, x, condition):
        h = self.norm1(self.conv1(x))
        scale, shift = self.cond(F.silu(condition)).chunk(2, dim=-1)
        h = F.silu(h * (1 + scale[:, :, None]) + shift[:, :, None])
        h = self.norm2(self.conv2(h))
        return F.silu(h + self.skip(x))


class TinyDiffusionPolicy(nn.Module):
    def __init__(self, obs_dim=8, action_dim=3, base=16, cond_dim=32):
        super().__init__()
        self.cond_dim = cond_dim
        self.obs_cond = nn.Linear(obs_dim, cond_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(cond_dim, cond_dim), nn.SiLU(), nn.Linear(cond_dim, cond_dim)
        )

        self.in_proj = nn.Conv1d(action_dim, base, 1)
        self.down = CondRes1D(base, base, cond_dim)
        self.downsample = nn.Conv1d(base, 2 * base, 4, stride=2, padding=1)
        self.mid = CondRes1D(2 * base, 2 * base, cond_dim)
        self.upsample = nn.ConvTranspose1d(2 * base, base, 4, stride=2, padding=1)
        self.up = CondRes1D(2 * base, base, cond_dim)
        self.out = nn.Conv1d(base, action_dim, 1)

    def forward(self, noisy_actions, obs, t):
        condition = self.obs_cond(obs) + self.time_mlp(time_embedding(t, self.cond_dim))
        x = self.in_proj(noisy_actions.transpose(1, 2))
        skip = self.down(x, condition)
        x = self.mid(self.downsample(skip), condition)
        x = self.upsample(x)

        if x.size(-1) != skip.size(-1):
            x = F.interpolate(x, size=skip.size(-1), mode="linear", align_corners=False)

        x = self.up(torch.cat([x, skip], dim=1), condition)
        return self.out(x).transpose(1, 2)


diffusion_policy = TinyDiffusionPolicy().to(device)
action_horizon = 8
clean_actions = torch.randn(4, action_horizon, 3, device=device)
noise = torch.randn_like(clean_actions)
t = torch.rand(4, device=device)

alpha = torch.cos(0.5 * math.pi * t)[:, None, None]
sigma = torch.sin(0.5 * math.pi * t)[:, None, None]
noisy_actions = alpha * clean_actions + sigma * noise

pred_noise = diffusion_policy(noisy_actions, obs, t)
diffusion_loss = F.mse_loss(pred_noise, noise)
diffusion_loss.backward()

print("Diffusion horizon:", pred_noise.shape)
print("U-Net mid grad:", diffusion_policy.mid.conv1.weight.grad.norm().item())


### Diffusion Policy sampling

Noise action horizon에서 시작해 여러 denoising step을 거친 뒤,
앞부분 action만 실행하고 다음 observation에서 다시 계획하는 receding-horizon 방식을 쓴다.


In [ ]:
@torch.no_grad()
def sample_diffusion_actions(model, obs, horizon=8, action_dim=3, steps=6):
    x = torch.randn(obs.size(0), horizon, action_dim, device=obs.device)

    for step in range(steps, 0, -1):
        t_now = torch.full((obs.size(0),), step / steps, device=obs.device)
        t_next = torch.full((obs.size(0),), (step - 1) / steps, device=obs.device)

        eps = model(x, obs, t_now)
        a_now = torch.cos(0.5 * math.pi * t_now)[:, None, None].clamp_min(1e-3)
        s_now = torch.sin(0.5 * math.pi * t_now)[:, None, None]
        x0_hat = (x - s_now * eps) / a_now

        a_next = torch.cos(0.5 * math.pi * t_next)[:, None, None]
        s_next = torch.sin(0.5 * math.pi * t_next)[:, None, None]
        x = a_next * x0_hat + s_next * eps

    return x


sampled_actions = sample_diffusion_actions(diffusion_policy, obs[:1])
print("sampled horizon:", sampled_actions.shape)
print("execute prefix:", sampled_actions[:, :2].shape)


## 3. π0-style action expert — separate parameters + joint masked attention

π0의 핵심을 `같은 generic Transformer에 전부 넣기`로 축소하지 않는다.

- vision/language prefix는 **base/VLM parameter set**
- robot state + noisy action suffix는 **action-expert parameter set**
- Q/K/V를 합쳐 joint attention
- prefix는 suffix를 볼 수 없고 suffix는 prefix를 볼 수 있는 asymmetric mask
- action expert가 noisy action horizon의 velocity를 직접 예측

따라서 sampling에도 oracle velocity를 쓰지 않는다.


In [ ]:
def pi0_mask(prefix_len, suffix_len, device):
    total = prefix_len + suffix_len
    mask = torch.zeros(total, total, dtype=torch.bool, device=device)
    mask[:prefix_len, :prefix_len] = True
    mask[prefix_len:, :] = True
    return mask


class AdaRMSNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.RMSNorm(dim)
        self.scale = nn.Linear(dim, dim)
        self.shift = nn.Linear(dim, dim)

    def forward(self, x, condition):
        return (
            self.norm(x) * (1 + self.scale(condition)[:, None])
            + self.shift(condition)[:, None]
        )


class TinyPi0JointBlock(nn.Module):
    def __init__(self, dim=24, heads=3):
        super().__init__()
        self.heads = heads
        self.head_dim = dim // heads

        # Base/VLM stream parameters.
        self.prefix_norm = nn.RMSNorm(dim)
        self.prefix_qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.prefix_out = nn.Linear(dim, dim, bias=False)
        self.prefix_mlp = nn.Sequential(
            nn.RMSNorm(dim), nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim)
        )

        # Action-expert stream parameters.
        self.expert_norm = AdaRMSNorm(dim)
        self.expert_qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.expert_out = nn.Linear(dim, dim, bias=False)
        self.expert_mlp_norm = AdaRMSNorm(dim)
        self.expert_mlp = nn.Sequential(
            nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim)
        )

    def _qkv(self, x, projection):
        batch, length, dim = x.shape
        qkv = projection(x).view(batch, length, 3, self.heads, self.head_dim)
        return qkv.permute(2, 0, 3, 1, 4).unbind(0)

    def forward(self, prefix, suffix, condition):
        pq, pk, pv = self._qkv(self.prefix_norm(prefix), self.prefix_qkv)
        eq, ek, ev = self._qkv(self.expert_norm(suffix, condition), self.expert_qkv)

        q = torch.cat([pq, eq], dim=2)
        k = torch.cat([pk, ek], dim=2)
        v = torch.cat([pv, ev], dim=2)

        mask = pi0_mask(prefix.size(1), suffix.size(1), prefix.device)
        attended = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)

        p_len = prefix.size(1)
        p_attn = attended[:, :, :p_len].transpose(1, 2).contiguous().flatten(2)
        e_attn = attended[:, :, p_len:].transpose(1, 2).contiguous().flatten(2)

        prefix = prefix + self.prefix_out(p_attn)
        suffix = suffix + self.expert_out(e_attn)

        prefix = prefix + self.prefix_mlp(prefix)
        suffix = suffix + self.expert_mlp(self.expert_mlp_norm(suffix, condition))
        return prefix, suffix, mask


class TinyPi0FlowPolicy(nn.Module):
    def __init__(self, vocab=32, vision_dim=10, state_dim=6, action_dim=3, horizon=6, dim=24):
        super().__init__()
        self.horizon = horizon
        self.action_dim = action_dim
        self.dim = dim

        self.vision_proj = nn.Linear(vision_dim, dim)
        self.language_embed = nn.Embedding(vocab, dim)

        self.state_proj = nn.Linear(state_dim, dim)
        self.action_proj = nn.Linear(action_dim, dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, dim)
        )

        self.blocks = nn.ModuleList([TinyPi0JointBlock(dim=dim, heads=3) for _ in range(2)])
        self.action_out = nn.Linear(dim, action_dim)

    def forward(self, vision, language, state, noisy_actions, t):
        prefix = torch.cat(
            [self.vision_proj(vision), self.language_embed(language)],
            dim=1,
        )

        time_condition = self.time_mlp(time_embedding(t, self.dim))
        state_token = self.state_proj(state).unsqueeze(1)
        action_tokens = self.action_proj(noisy_actions) + time_condition[:, None]
        suffix = torch.cat([state_token, action_tokens], dim=1)

        mask = None
        for block in self.blocks:
            prefix, suffix, mask = block(prefix, suffix, time_condition)

        return self.action_out(suffix[:, 1:]), mask


pi0 = TinyPi0FlowPolicy().to(device)
vision = torch.randn(3, 4, 10, device=device)
language = torch.randint(0, 32, (3, 5), device=device)
state = torch.randn(3, 6, device=device)

expert_actions = torch.randn(3, 6, 3, device=device)
noise_actions = torch.randn_like(expert_actions)
t = torch.rand(3, device=device)

x_t = t[:, None, None] * noise_actions + (1 - t[:, None, None]) * expert_actions
target_velocity = noise_actions - expert_actions

pred_velocity, mask = pi0(vision, language, state, x_t, t)
pi0_loss = F.mse_loss(pred_velocity, target_velocity)
pi0_loss.backward()

print("π0 velocity:", pred_velocity.shape)
print("prefix -> action allowed:", bool(mask[0, -1]))
print("action -> prefix allowed:", bool(mask[-1, 0]))
print("base QKV grad:", pi0.blocks[0].prefix_qkv.weight.grad.norm().item())
print("expert QKV grad:", pi0.blocks[0].expert_qkv.weight.grad.norm().item())


### π0-style ODE sampling

학습된 action expert가 예측한 velocity를 직접 적분한다.
`expert_actions`를 참조하는 oracle function은 사용하지 않는다.


In [ ]:
@torch.no_grad()
def sample_pi0(model, vision, language, state, steps=8):
    x = torch.randn(
        vision.size(0), model.horizon, model.action_dim, device=vision.device
    )

    dt = -1.0 / steps
    for step in range(steps):
        t = torch.full((vision.size(0),), 1.0 - step / steps, device=vision.device)
        velocity, _ = model(vision, language, state, x, t)
        x = x + dt * velocity

    return x


pi0_sample = sample_pi0(pi0, vision[:1], language[:1], state[:1])
print("π0 sampled horizon:", pi0_sample.shape)


## 4. FAST — quantile normalization → DCT → quantization

FAST는 raw action을 바로 scalar bin으로 만들지 않는다.

먼저 action dimension별 quantile normalization을 하고,
time axis DCT coefficient를 quantize한 뒤 low-frequency-first integer sequence를 만든다.


In [ ]:
action_dataset = torch.randn(64, 8, 3, device=device)
flat = action_dataset.reshape(-1, action_dataset.size(-1))
q_low = torch.quantile(flat, 0.01, dim=0)
q_high = torch.quantile(flat, 0.99, dim=0)


def normalize_actions(actions):
    value = 2 * (actions - q_low) / (q_high - q_low + 1e-6) - 1
    return value.clamp(-1, 1)


def dct_matrix(length, device):
    n = torch.arange(length, device=device, dtype=torch.float32)
    k = torch.arange(length, device=device, dtype=torch.float32)[:, None]
    matrix = torch.cos(math.pi / length * (n[None] + 0.5) * k)
    matrix[0] *= 1 / math.sqrt(length)
    matrix[1:] *= math.sqrt(2 / length)
    return matrix


trajectory = normalize_actions(action_dataset[0])
DCT = dct_matrix(trajectory.size(0), device)
coefficients = DCT @ trajectory

quantization_scale = 32.0
quantized = torch.round(coefficients * quantization_scale).to(torch.int64)
integer_sequence = quantized.reshape(-1).tolist()

print("normalized range:", float(trajectory.min()), float(trajectory.max()))
print("DCT shape:", coefficients.shape)
print("integer sequence length:", len(integer_sequence))


## 5. FAST — lossless BPE compression

실제 FAST는 DCT integer sequence에 BPE를 학습해 action-token sequence를 더 짧게 만든다.
아래 tiny BPE는 같은 아이디어를 integer symbols에 직접 적용하고 decode 후 완전 복원을 확인한다.


In [ ]:
def count_pairs(sequences):
    counts = Counter()
    for sequence in sequences:
        counts.update(zip(sequence[:-1], sequence[1:]))
    return counts


def merge_pair(sequence, pair, new_symbol):
    output = []
    i = 0

    while i < len(sequence):
        if i + 1 < len(sequence) and (sequence[i], sequence[i + 1]) == pair:
            output.append(new_symbol)
            i += 2
        else:
            output.append(sequence[i])
            i += 1

    return output


def train_bpe(sequences, num_merges=12):
    sequences = [list(sequence) for sequence in sequences]
    maximum = max(abs(symbol) for seq in sequences for symbol in seq)
    next_symbol = maximum + 1
    rules = []

    for _ in range(num_merges):
        counts = count_pairs(sequences)
        if not counts:
            break

        pair, count = max(counts.items(), key=lambda item: item[1])
        if count < 2:
            break

        rules.append((pair, next_symbol))
        sequences = [merge_pair(seq, pair, next_symbol) for seq in sequences]
        next_symbol += 1

    return rules


def encode_bpe(sequence, rules):
    encoded = list(sequence)
    for pair, symbol in rules:
        encoded = merge_pair(encoded, pair, symbol)
    return encoded


def decode_bpe(sequence, rules):
    expansion = {symbol: pair for pair, symbol in rules}

    def expand(symbol):
        if symbol not in expansion:
            return [symbol]
        left, right = expansion[symbol]
        return expand(left) + expand(right)

    output = []
    for symbol in sequence:
        output.extend(expand(symbol))
    return output


corpus = []
for sample in action_dataset[:16]:
    coeff = DCT @ normalize_actions(sample)
    symbols = torch.round(coeff * quantization_scale).to(torch.int64)
    corpus.append(symbols.reshape(-1).tolist())

rules = train_bpe(corpus)
encoded = encode_bpe(integer_sequence, rules)
decoded = decode_bpe(encoded, rules)

print("BPE merges:", len(rules))
print("before / after:", len(integer_sequence), len(encoded))
print("lossless recovery:", decoded == integer_sequence)


## References and provenance

**ACT** — action chunk + CVAE latent + Transformer query decoding을 유지한다.

**Diffusion Policy** — observation-conditioned temporal diffusion과 conditional 1D U-Net의 down/mid/up/skip 구조를 유지하고 channel 수만 줄였다.

**π0 / openpi** — vision-language prefix와 별도의 action-expert parameter set, asymmetric prefix/suffix attention, time-conditioned flow velocity prediction과 iterative integration을 반영했다.

**FAST** — quantile normalization, temporal DCT, coefficient quantization, frequency-major integer sequence, BPE compression을 모두 포함한다.

이 노트북은 benchmark 재현이 아니라 **핵심 구조에 실제 gradient가 흐르는 최소구현**을 목표로 한다.
